# Custom Embeddings with NVIDIA Nemotron (NV-Embed-v2) & Fine-Tuning Setup

This notebook demonstrates how to load your generated training datasets, apply NVIDIA Nemotron `NV-Embed-v2` embeddings for vector retrieval, and prepare NVIDIA Nemotron-4-34B / Llama-3-Nemotron fine-tuning configurations.

In [ ]:
import os
import json
from scripts.custom_embed import prepare_documents_from_jsonl, format_instruction_for_nv_embed, batch_documents

# Model targets
NV_EMBED_MODEL = "nvidia/NV-Embed-v2"
NEMOTRON_SFT_MODEL = "nvidia/Nemotron-4-34B-Instruct"

print(f"NVIDIA NV-Embed Model Target: {NV_EMBED_MODEL}")
print(f"NVIDIA Nemotron SFT Target: {NEMOTRON_SFT_MODEL}")

In [ ]:
# Load JSONL documents generated previously
jsonl_path = "tests/test_data.jsonl"
if not os.path.exists(jsonl_path):
    # Generate temporary sample for notebook run
    os.makedirs("tests", exist_ok=True)
    sample_data = [
        {"messages": [{"role": "user", "content": "How does Nemotron NV-Embed-v2 work?"}, {"role": "assistant", "content": "NV-Embed-v2 is an elite embedding model achieving top rank on MTEB using latent attention pooling."}]},
        {"text": "File: system.py\n\nimport os\nprint('Initialized Nemotron Pipeline')"}
    ]
    with open(jsonl_path, "w") as f:
        for s in sample_data:
            f.write(json.dumps(s) + "\n")

docs = prepare_documents_from_jsonl(jsonl_path)
print(f"Loaded {len(docs)} documents for NV-Embed processing.")

In [ ]:
# Format inputs with NV-Embed-v2 required retrieval instructions
formatted_prompts = []
for doc in docs:
    prompt = format_instruction_for_nv_embed(doc["text"])
    formatted_prompts.append(prompt)

print("Sample Formatted NV-Embed Prompt:")
print(formatted_prompts[0][:300])

In [ ]:
# PyTorch / HuggingFace NV-Embed-v2 Integration Snippet
nv_embed_code_snippet = '''
import torch
from transformers import AutoModel

# Load NV-Embed-v2 with trust_remote_code=True
model = AutoModel.from_pretrained('nvidia/NV-Embed-v2', trust_remote_code=True, torch_dtype=torch.bfloat16)

embeddings = model.encode(
    prompts=formatted_prompts,
    instruction="Instruct: Retrieve relevant documentation and conversation context\nQuery: ",
    max_length=32768,
)
'''
print("NV-Embed-v2 PyTorch loading code ready:")
print(nv_embed_code_snippet)